<a href="https://colab.research.google.com/github/Rohan-724/CSA1702/blob/main/Lab/Exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 8-Puzzle Solver using A* Search

The 8-Puzzle is a classic sliding puzzle that consists of a 3x3 grid with 8 numbered tiles and one blank space. The goal is to rearrange the tiles to a specific goal configuration by sliding them into the blank space.

We will use the A* search algorithm to find the shortest sequence of moves to reach the goal state. A* search requires a heuristic function to estimate the cost from the current state to the goal. For the 8-Puzzle, the Manhattan distance is a commonly used and admissible heuristic.

In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", cost=0):
        self.board = board  # Tuple of tuples for immutability (e.g., ((1,2,3),(4,5,6),(7,8,0)))
        self.parent = parent
        self.move = move
        self.cost = cost  # Cost from start to current state (g(n))
        self.heuristic = self._calculate_manhattan_distance() # Heuristic cost (h(n))
        self.f_cost = self.cost + self.heuristic # f(n) = g(n) + h(n)
        self.blank_pos = self._find_blank()

    def __lt__(self, other):
        return self.f_cost < other.f_cost

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

    def _find_blank(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return (r, c)
        return None

    def _calculate_manhattan_distance(self):
        distance = 0
        for r in range(3):
            for c in range(3):
                tile = self.board[r][c]
                if tile == 0:
                    continue
                target_r, target_c = divmod(tile - 1, 3)
                distance += abs(r - target_r) + abs(c - target_c)
        return distance

    def get_neighbors(self):
        neighbors = []
        br, bc = self.blank_pos
        moves = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}

        for move_name, (dr, dc) in moves.items():
            new_br, new_bc = br + dr, bc + dc

            if 0 <= new_br < 3 and 0 <= new_bc < 3:
                new_board_list = [list(row) for row in self.board]
                # Swap blank with the tile at (new_br, new_bc)
                new_board_list[br][bc], new_board_list[new_br][new_bc] = \
                    new_board_list[new_br][new_bc], new_board_list[br][bc]
                new_board_tuple = tuple(tuple(row) for row in new_board_list)
                neighbors.append(PuzzleState(new_board_tuple, self, move_name, self.cost + 1))
        return neighbors

    def print_board(self):
        for row in self.board:
            print(" ".join(map(str, row)).replace('0', ' '))
        print("-" * 5)


In [ ]:
def solve_8_puzzle(initial_board, goal_board):
    initial_state = PuzzleState(initial_board)
    goal_state = PuzzleState(goal_board)

    # Priority queue to store states to be explored (ordered by f_cost)
    open_list = [initial_state]
    heapq.heapify(open_list)

    # Set to store visited states to avoid cycles
    closed_list = set()

    while open_list:
        current_state = heapq.heappop(open_list)

        if current_state == goal_state:
            path = []
            while current_state.parent:
                path.append(current_state.move)
                current_state = current_state.parent
            return path[::-1]  # Return reversed path

        closed_list.add(current_state)

        for neighbor in current_state.get_neighbors():
            if neighbor in closed_list:
                continue

            # Check if neighbor is already in open_list with a higher cost
            found_in_open = False
            for i, item in enumerate(open_list):
                if item == neighbor and item.cost > neighbor.cost:
                    # Update cost and re-heapify
                    open_list[i] = neighbor
                    heapq.heapify(open_list)
                    found_in_open = True
                    break
                elif item == neighbor and item.cost <= neighbor.cost:
                    found_in_open = True
                    break

            if not found_in_open:
                heapq.heappush(open_list, neighbor)

    return None  # No solution found


In [ ]:
# Define the initial and goal states
# 0 represents the blank space
initial_board = (
    (1, 2, 3),
    (0, 4, 6),
    (7, 5, 8)
)

goal_board = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

print("Initial Board:")
PuzzleState(initial_board).print_board()

print("Goal Board:")
PuzzleState(goal_board).print_board()

print("Solving...")
solution_path = solve_8_puzzle(initial_board, goal_board)

if solution_path:
    print(f"Solution found in {len(solution_path)} moves:")
    current_board_state = initial_board
    for move in solution_path:
        print(f"Move: {move}")
        # Simulate the move for printing purposes
        temp_state = PuzzleState(current_board_state)
        for neighbor in temp_state.get_neighbors():
            if neighbor.move == move:
                current_board_state = neighbor.board
                break
        PuzzleState(current_board_state).print_board()
else:
    print("No solution found.")
